# Phase 03C: Teacher A ExpansionNet v2 Pseudo-Caption Readiness

Phase 03B đã chứng minh checkpoint ExpansionNet v2 chính thức đạt
`CIDEr-D = 1.397207` trên 5.000 ảnh Karpathy Validation và vượt Teacher A
CLIP cũ `+0.236743`.

Checkpoint này đã được huấn luyện đầy đủ và tối ưu bằng reinforcement learning.
Vì vậy Phase 03C **không huấn luyện lại Teacher**. Mục tiêu đúng là khóa
production engine để sinh pseudo-caption cho 113.287 ảnh Train+Restval phục vụ
Sequence KD cho S1 MobileNetV3-Large.

## Phạm vi

- xác minh toàn bộ Phase 03B bằng checksum chain;
- dựng Train+Restval image index, không đọc caption huấn luyện;
- khóa ánh xạ `Teacher words → Phase 1 tokenizer → S1 token IDs`;
- strict-load lại frozen ExpansionNet v2;
- benchmark batch size 1/2/4/8;
- chạy discarded pilot 64 ảnh và kiểm exact repeat;
- khóa sharding/resume/atomic-output contract cho Phase 03D;
- Held-out Test luôn `SEALED`.

## Kaggle inputs

1. Phase 1 Formal Output.
2. Phase 2A Shared Student Pipeline.
3. Phase 03B ExpansionNet v2 completed output.
4. MS COCO 2014 images.
5. ExpansionNet v2 offline model package.

GPU T4 hoặc mạnh hơn, Internet Off, chạy Cell 1 → Cell 7.


## Cell 1 — Authorization, integrity và input discovery tối ưu

Chỉ tìm marker trong `/kaggle/input/notebooks` và model trong
`/kaggle/input/models`; không quét đệ quy toàn bộ 123.287 ảnh COCO.

**Đầu ra dự kiến:** Phase 03B manifest `14/14 VERIFIED`, CIDEr-D `1.397207`,
Test `SEALED`, tiến độ `1/7`.


In [1]:
import csv
import gc
import hashlib
import json
import math
import os
import pickle
import platform
import random
import re
import sys
import time
import unicodedata
from argparse import Namespace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 42
INPUT_ROOT = Path("/kaggle/input")
NOTEBOOK_INPUT_ROOT = INPUT_ROOT / "notebooks"
MODEL_INPUT_ROOT = INPUT_ROOT / "models"
DATASET_INPUT_ROOT = INPUT_ROOT / "datasets"
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase3_teacher_a_expansionnet_v2_readiness"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
PILOT_DIR = OUTPUT_ROOT / "pilot"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, PILOT_DIR, MANIFEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one_under(root, name):
    matches = sorted(Path(root).rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {root}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA không khả dụng. Hãy chọn GPU T4 hoặc mạnh hơn."
    )
set_deterministic(SEED)
device = torch.device("cuda:0")

PHASE3B_READY_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT,
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json",
)
PHASE3B_ROOT = PHASE3B_READY_PATH.parent
PHASE1_READY_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT, "PHASE1_FORMAL_DELIVERABLES_READY.json"
)
PHASE2_READY_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT, "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
phase3b_ready = read_json(PHASE3B_READY_PATH)
assert phase3b_ready["status"] == "ready"
assert phase3b_ready["teacher"] == "ExpansionNet v2"
assert phase3b_ready["matched_student"] == "S1 MobileNetV3-Large"
assert phase3b_ready["quality_gate_passed"] is True
assert phase3b_ready["direct_logit_kd_allowed"] is False
assert phase3b_ready["sequence_kd_required"] is True
assert phase3b_ready["phase03c_authorized"] is True
assert phase3b_ready["test_set_status"] == "sealed"
assert phase3b_ready["test_set_used"] is False

manifest_ref = phase3b_ready["manifest"]["csv"]
sums_ref = phase3b_ready["manifest"]["sha256sums"]
PHASE3B_MANIFEST = resolve_inside(
    PHASE3B_ROOT, manifest_ref["relative_path"]
)
PHASE3B_SUMS = resolve_inside(
    PHASE3B_ROOT, sums_ref["relative_path"]
)
assert sha256_file(PHASE3B_MANIFEST) == manifest_ref["sha256"]
assert sha256_file(PHASE3B_SUMS) == sums_ref["sha256"]
with PHASE3B_MANIFEST.open(
    "r", encoding="utf-8", newline=""
) as handle:
    phase3b_rows = list(csv.DictReader(handle))
assert len(phase3b_rows) == 14
for row in phase3b_rows:
    artifact = resolve_inside(PHASE3B_ROOT, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

for key in [
    "architecture_contract",
    "compatibility_contract",
    "sequence_kd_contract",
]:
    reference = phase3b_ready[key]
    artifact = resolve_inside(
        PHASE3B_ROOT, reference["relative_path"]
    )
    assert sha256_file(artifact) == reference["sha256"]

write_json(
    AUDIT_DIR / "phase3c_cell01_authorization.json",
    {
        "audit_version": "phase3c_expansionnet_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "phase3b_manifest_rows": len(phase3b_rows),
        "phase3b_validation_metrics": (
            phase3b_ready["validation_metrics"]
        ),
        "strict_deterministic": True,
        "optimized_input_discovery": True,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03C EXPANSIONNET V2 CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 03B manifest        : 14 / 14 VERIFIED")
print(
    "Phase 03B CIDEr-D        :",
    phase3b_ready["validation_metrics"]["CIDEr-D"],
)
print("Quality gate              : PASSED")
print("Sequence KD               : REQUIRED")
print("Optimized input discovery : True")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 7 code cells")
print("Next                      : Train+Restval image index")


PHASE 03C EXPANSIONNET V2 CELL 1 STATUS
Status                    : PASSED
Phase 03B manifest        : 14 / 14 VERIFIED
Phase 03B CIDEr-D        : 1.397207039078058
Quality gate              : PASSED
Sequence KD               : REQUIRED
Optimized input discovery : True
GPU                       : Tesla T4
Training started          : False
Test Set status           : SEALED
Progress                  : 1 / 7 code cells
Next                      : Train+Restval image index


## Cell 2 — Train+Restval image index, không đọc training captions

Khóa đúng 113.287 ảnh train; chỉ đọc image manifest. Held-out Test chỉ lấy
ID để kiểm overlap, không mở ảnh/caption/Test Parquet.


In [2]:
CELL_STARTED_AT = time.perf_counter()
import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

IMAGE_MANIFEST_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT, "mscoco_image_manifest.parquet"
)


def find_coco_leaf(relative_parts, probe_prefix):
    candidates = []
    for owner in DATASET_INPUT_ROOT.iterdir():
        candidate = owner.joinpath(*relative_parts)
        if (
            candidate.is_dir()
            and next(candidate.glob(f"{probe_prefix}*.jpg"), None)
        ):
            candidates.append(candidate.resolve())
    assert len(candidates) == 1, candidates
    return candidates[0]


TRAIN_LEAF = find_coco_leaf(
    ("mscoco2014-1", "train2014", "train2014"),
    "COCO_train2014_",
)
VAL_LEAF = find_coco_leaf(
    ("mscoco2014-1", "val2014", "val2014"),
    "COCO_val2014_",
)
manifest_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=[
        "image_id", "experiment_split", "source_partition", "image_path"
    ],
)
TRAIN_PATHS = {}
VALIDATION_IDS = set()
TEST_IDS = set()
for row in manifest_table.to_pylist():
    image_id = int(row["image_id"])
    split = row["experiment_split"]
    if split == "held_out_test":
        TEST_IDS.add(image_id)
        continue
    if split == "validation":
        VALIDATION_IDS.add(image_id)
        continue
    if split != "train":
        continue
    leaf = (
        TRAIN_LEAF
        if row["source_partition"] == "train2014"
        else VAL_LEAF
    )
    TRAIN_PATHS[image_id] = leaf / Path(row["image_path"]).name

assert len(TRAIN_PATHS) == 113287
assert len(VALIDATION_IDS) == 5000
assert len(TEST_IDS) == 5000
assert set(TRAIN_PATHS).isdisjoint(VALIDATION_IDS)
assert set(TRAIN_PATHS).isdisjoint(TEST_IDS)
assert VALIDATION_IDS.isdisjoint(TEST_IDS)
TRAIN_IMAGE_IDS = sorted(TRAIN_PATHS)
for image_id in TRAIN_IMAGE_IDS[:256]:
    assert TRAIN_PATHS[image_id].is_file()

PREPROCESS = transforms.Compose(
    [
        transforms.Resize(
            (384, 384), interpolation=InterpolationMode.BILINEAR
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


class TrainImages(Dataset):
    def __len__(self):
        return len(TRAIN_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TRAIN_IMAGE_IDS[index]
        with Image.open(TRAIN_PATHS[image_id]) as source:
            image = PREPROCESS(source.convert("RGB"))
        return {"image_id": image_id, "image": image}


TRAIN_DATASET = TrainImages()
sample_a = TRAIN_DATASET[0]["image"]
sample_b = TRAIN_DATASET[0]["image"]
assert sample_a.shape == (3, 384, 384)
assert torch.equal(sample_a, sample_b)
assert torch.isfinite(sample_a).all()

write_json(
    AUDIT_DIR / "phase3c_cell02_train_image_index.json",
    {
        "audit_version": "phase3c_expansionnet_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "train_restval_images": 113287,
        "validation_ids": 5000,
        "held_out_test_ids": 5000,
        "membership_overlaps": 0,
        "path_probes_passed": 256,
        "preprocessing_repeat_exact": True,
        "training_caption_read": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03C EXPANSIONNET V2 CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Train+Restval images      : 113287")
print("Validation IDs            : 5000")
print("Held-out Test IDs         : 5000")
print("Membership overlaps       : 0")
print("Image probes              : 256 / 256 PASSED")
print("Training caption read     : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 7 code cells")
print("Next                      : lock Teacher-to-S1 token conversion")


PHASE 03C EXPANSIONNET V2 CELL 2 STATUS
Status                    : PASSED
Train+Restval images      : 113287
Validation IDs            : 5000
Held-out Test IDs         : 5000
Membership overlaps       : 0
Image probes              : 256 / 256 PASSED
Training caption read     : False
Test Parquet opened       : False
Test images read          : False
Test Set status           : SEALED
Progress                  : 2 / 7 code cells
Next                      : lock Teacher-to-S1 token conversion


## Cell 3 — Teacher words → Phase 1 tokenizer → S1 token IDs

Kiểm tra conversion trên toàn bộ 5.000 prediction Validation đã sinh ở
Phase 03B. Chuỗi Student luôn có BOS/EOS, tối đa 30 lexical token và dùng UNK
theo contract Phase 1.


In [3]:
CELL_STARTED_AT = time.perf_counter()

VOCAB_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT, "word_vocabulary_minfreq5.json"
)
TOKENIZER_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT, "tokenizer_contract.json"
)
SEQUENCE_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT, "sequence_encoding_contract.json"
)
PHASE3B_PREDICTIONS = resolve_inside(
    PHASE3B_ROOT,
    "validation/expansionnet_v2_validation_predictions.json",
)
vocabulary = read_json(VOCAB_PATH)
tokenizer_contract = read_json(TOKENIZER_PATH)
sequence_contract = read_json(SEQUENCE_PATH)
validation_predictions = read_json(PHASE3B_PREDICTIONS)
TOKEN_TO_ID = {
    token: index
    for index, token in enumerate(vocabulary["id_to_token"])
}
assert len(TOKEN_TO_ID) == 10080
assert tokenizer_contract["token_pattern"] == (
    r"[a-z0-9]+(?:'[a-z0-9]+)?"
)
assert sequence_contract["length_policy"][
    "maximum_lexical_tokens"
] == 30
TOKEN_PATTERN = re.compile(tokenizer_contract["token_pattern"])


def phase1_tokenize(text):
    normalized = unicodedata.normalize("NFKC", text).lower()
    normalized = (
        normalized.replace("’", "'")
        .replace("‘", "'")
        .replace("`", "'")
    )
    return TOKEN_PATTERN.findall(normalized)


def caption_to_s1(caption):
    lexical = phase1_tokenize(caption)[:30]
    ids = [1] + [TOKEN_TO_ID.get(token, 3) for token in lexical] + [2]
    assert 2 <= len(ids) <= 32
    return lexical, ids


total_lexical = 0
total_unk = 0
lengths = []
for caption in validation_predictions.values():
    lexical, token_ids = caption_to_s1(caption)
    total_lexical += len(lexical)
    total_unk += sum(token_id == 3 for token_id in token_ids)
    lengths.append(len(lexical))
assert len(validation_predictions) == 5000
assert max(lengths) <= 30
unk_rate = total_unk / max(1, total_lexical)

TOKEN_MAPPING_CONTRACT = (
    CONFIG_DIR / "teacher_a_expansionnet_v2_to_s1_token_mapping.json"
)
write_json(
    TOKEN_MAPPING_CONTRACT,
    {
        "contract_version": "teacher_a_expansionnet_to_s1_mapping_v1",
        "created_at_utc": now_utc(),
        "normalization": tokenizer_contract["normalization_steps"],
        "token_pattern": tokenizer_contract["token_pattern"],
        "maximum_lexical_tokens": 30,
        "student_special_ids": {
            "pad": 0, "bos": 1, "eos": 2, "unk": 3
        },
        "conversion": (
            "teacher IDs -> native words -> normalized caption -> "
            "Phase 1 regex tokenizer -> S1 IDs"
        ),
        "validation_predictions_checked": 5000,
        "validation_lexical_tokens": total_lexical,
        "validation_unknown_tokens": total_unk,
        "validation_unknown_rate": unk_rate,
        "test_set_status": "sealed",
    },
)
write_json(
    AUDIT_DIR / "phase3c_cell03_token_mapping.json",
    {
        "audit_version": "phase3c_expansionnet_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "predictions_checked": 5000,
        "lexical_tokens": total_lexical,
        "unknown_tokens": total_unk,
        "unknown_rate": unk_rate,
        "maximum_observed_lexical_length": max(lengths),
        "all_sequences_have_bos_eos": True,
        "direct_logit_kd_allowed": False,
        "sequence_kd_mapping_ready": True,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03C EXPANSIONNET V2 CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Validation captions mapped: 5000 / 5000")
print("Lexical tokens            :", total_lexical)
print("Unknown tokens            :", total_unk)
print("Unknown-token rate        :", f"{unk_rate:.6%}")
print("Maximum lexical length    :", max(lengths), "/ 30")
print("BOS/EOS policy            : PASSED")
print("Direct Logit KD           : PROHIBITED")
print("Sequence mapping          : READY")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 7 code cells")
print("Next                      : strict-load frozen ExpansionNet v2")


PHASE 03C EXPANSIONNET V2 CELL 3 STATUS
Status                    : PASSED
Validation captions mapped: 5000 / 5000
Lexical tokens            : 47645
Unknown tokens            : 0
Unknown-token rate        : 0.000000%
Maximum lexical length    : 16 / 30
BOS/EOS policy            : PASSED
Direct Logit KD           : PROHIBITED
Sequence mapping          : READY
Test Set status           : SEALED
Progress                  : 3 / 7 code cells
Next                      : strict-load frozen ExpansionNet v2


## Cell 4 — Reconstruct và strict-load frozen ExpansionNet v2

Không lặp lại SHA-256 2,82 GB: provenance được nối từ marker Phase 03B và
manifest model. Cell này xác minh cùng model version/kích thước rồi strict-load
state dict.


In [4]:
CELL_STARTED_AT = time.perf_counter()

model_manifests = []
for candidate in MODEL_INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        payload = read_json(candidate)
    except Exception:
        continue
    if payload.get("model_id") == "expansionnet_v2":
        model_manifests.append((candidate.resolve(), payload))
assert len(model_manifests) == 1, model_manifests
MODEL_MANIFEST_PATH, model_manifest = model_manifests[0]
MODEL_ROOT = MODEL_MANIFEST_PATH.parent
checkpoint_record = model_manifest["checkpoint"]
CHECKPOINT_PATH = MODEL_ROOT / checkpoint_record["relative_path"]
assert CHECKPOINT_PATH.is_file()
assert CHECKPOINT_PATH.stat().st_size == int(
    checkpoint_record["size_bytes"]
)
assert checkpoint_record["sha256"] == (
    phase3b_ready["checkpoint_sha256"]
)
NATIVE_VOCAB_PATH = (
    MODEL_ROOT / model_manifest["vocabulary"]["relative_path"]
)
assert sha256_file(NATIVE_VOCAB_PATH) == (
    model_manifest["vocabulary"]["sha256"]
)
with NATIVE_VOCAB_PATH.open("rb") as handle:
    NATIVE_TOKENS = pickle.load(handle)
NATIVE_WORD2IDX = NATIVE_TOKENS["word2idx_dict"]
NATIVE_IDX2WORD = NATIVE_TOKENS["idx2word_list"]
NATIVE_BOS = int(NATIVE_WORD2IDX["SOS"])
NATIVE_EOS = int(NATIVE_WORD2IDX["EOS"])
NATIVE_PAD = int(NATIVE_WORD2IDX["PAD"])
assert len(NATIVE_WORD2IDX) == 10000

SOURCE_ROOT = MODEL_ROOT / "ExpansionNet_v2_src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

drop_args = Namespace(
    enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0
)
TEACHER = End_ExpansionNet_v2(
    swin_img_size=384, swin_patch_size=4, swin_in_chans=3,
    swin_embed_dim=192, swin_depths=[2, 2, 18, 2],
    swin_num_heads=[6, 12, 24, 48], swin_window_size=12,
    swin_mlp_ratio=4.0, swin_qkv_bias=True, swin_qk_scale=None,
    swin_drop_rate=0.0, swin_attn_drop_rate=0.0,
    swin_drop_path_rate=0.0, swin_norm_layer=torch.nn.LayerNorm,
    swin_ape=False, swin_patch_norm=True, swin_use_checkpoint=False,
    final_swin_dim=1536, d_model=512, N_enc=3, N_dec=3,
    num_heads=8, ff=2048,
    num_exp_enc_list=[32, 64, 128, 256, 512],
    num_exp_dec=16, output_word2idx=NATIVE_WORD2IDX,
    output_idx2word=NATIVE_IDX2WORD, max_seq_len=74,
    drop_args=drop_args, rank=device,
)
checkpoint = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
incompatible = TEACHER.load_state_dict(
    checkpoint["model_state_dict"], strict=True
)
assert not incompatible.missing_keys
assert not incompatible.unexpected_keys
del checkpoint
gc.collect()
TEACHER = TEACHER.to(device).eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad_(False)
assert sum(p.numel() for p in TEACHER.parameters()) == 233803076
assert sum(p.numel() for p in TEACHER.parameters() if p.requires_grad) == 0


def decode_native(tokens):
    words = []
    for token in tokens:
        token = int(token)
        if token == NATIVE_BOS:
            continue
        if token == NATIVE_EOS:
            break
        if token == NATIVE_PAD:
            continue
        words.append(str(NATIVE_IDX2WORD[token]))
    return " ".join(words).strip()


def generate_batch(images):
    images = images.to(device, non_blocking=True)
    with torch.inference_mode():
        predictions, log_probs = TEACHER(
            enc_x=images,
            enc_x_num_pads=[0] * images.shape[0],
            mode="beam_search",
            beam_size=3,
            beam_max_seq_len=31,
            sample_or_max="max",
            how_many_outputs=1,
            sos_idx=NATIVE_BOS,
            eos_idx=NATIVE_EOS,
            length_penalty=0.7,
        )
    tokens = [
        [int(value) for value in predictions[index][0]]
        for index in range(len(predictions))
    ]
    captions = [decode_native(values) for values in tokens]
    return tokens, captions, log_probs


write_json(
    AUDIT_DIR / "phase3c_cell04_model_load.json",
    {
        "audit_version": "phase3c_expansionnet_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "checkpoint_provenance_chained_to_phase3b": True,
        "checkpoint_size_verified": True,
        "checkpoint_rehash_skipped": True,
        "strict_load": True,
        "missing_keys": 0,
        "unexpected_keys": 0,
        "parameters": 233803076,
        "frozen_teacher": True,
        "optimizer_created": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03C EXPANSIONNET V2 CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Checkpoint provenance     : CHAINED TO PHASE 03B")
print("Redundant 2.82GB rehash   : SKIPPED")
print("Strict checkpoint load    : PASSED")
print("Missing/unexpected keys   : 0 / 0")
print("Parameters                : 233803076")
print("Frozen Teacher            : True")
print("Optimizer created         : False")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 7 code cells")
print("Next                      : batch-size benchmark")


PHASE 03C EXPANSIONNET V2 CELL 4 STATUS
Status                    : PASSED
Checkpoint provenance     : CHAINED TO PHASE 03B
Redundant 2.82GB rehash   : SKIPPED
Strict checkpoint load    : PASSED
Missing/unexpected keys   : 0 / 0
Parameters                : 233803076
Frozen Teacher            : True
Optimizer created         : False
Training started          : False
Test Set status           : SEALED
Progress                  : 4 / 7 code cells
Next                      : batch-size benchmark


## Cell 5 — Batch-size benchmark 1/2/4/8

Chọn batch lớn nhất chạy ổn định dưới 12 GiB peak allocation. Mỗi cấu hình
được chạy trên cùng ảnh và kiểm caption hữu hạn. OOM ở batch lớn chỉ làm loại
ứng viên đó, không làm mất checkpoint.


In [5]:
CELL_STARTED_AT = time.perf_counter()


def collate_samples(samples):
    return torch.stack([sample["image"] for sample in samples])


BENCHMARK_RESULTS = []
benchmark_samples = [TRAIN_DATASET[index] for index in range(8)]
for batch_size in [1, 2, 4, 8]:
    try:
        images = collate_samples(benchmark_samples[:batch_size])
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        start = time.perf_counter()
        tokens, captions, scores = generate_batch(images)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        assert len(captions) == batch_size
        assert all(captions)
        result = {
            "batch_size": batch_size,
            "status": "passed",
            "elapsed_seconds": elapsed,
            "seconds_per_image": elapsed / batch_size,
            "peak_gpu_memory_mib": (
                torch.cuda.max_memory_allocated() / (1024 ** 2)
            ),
        }
    except torch.cuda.OutOfMemoryError:
        result = {"batch_size": batch_size, "status": "oom"}
        torch.cuda.empty_cache()
    BENCHMARK_RESULTS.append(result)

passed = [
    row for row in BENCHMARK_RESULTS
    if row["status"] == "passed"
    and row["peak_gpu_memory_mib"] < 12 * 1024
]
assert passed
SELECTED_BATCH = max(row["batch_size"] for row in passed)
selected_result = next(
    row for row in passed if row["batch_size"] == SELECTED_BATCH
)

# Exact repeat on the selected batch.
selected_images = collate_samples(
    benchmark_samples[:SELECTED_BATCH]
)
tokens_a, captions_a, scores_a = generate_batch(selected_images)
tokens_b, captions_b, scores_b = generate_batch(selected_images)
assert tokens_a == tokens_b
assert captions_a == captions_b

write_json(
    CONFIG_DIR / "teacher_a_expansionnet_v2_batch_benchmark.json",
    {
        "contract_version": "phase3c_batch_benchmark_v1",
        "created_at_utc": now_utc(),
        "candidates": BENCHMARK_RESULTS,
        "selected_batch_size": SELECTED_BATCH,
        "selection_rule": (
            "largest passing batch with peak allocation below 12 GiB"
        ),
        "selected_batch_repeat_exact": True,
        "test_set_status": "sealed",
    },
)
write_json(
    AUDIT_DIR / "phase3c_cell05_batch_benchmark.json",
    {
        "audit_version": "phase3c_expansionnet_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "results": BENCHMARK_RESULTS,
        "selected_batch_size": SELECTED_BATCH,
        "selected_batch_repeat_exact": True,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03C EXPANSIONNET V2 CELL 5 STATUS")
print("=" * 72)
for row in BENCHMARK_RESULTS:
    print("Batch", row["batch_size"], ":", row["status"].upper(), row)
print("Selected batch size       :", SELECTED_BATCH)
print("Selected repeat exact     : True")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 7 code cells")
print("Next                      : discarded 64-image production pilot")


/kaggle/input/models/nguynthnhdan/expansionnet-v2-ieee-bigdata2023/pytorch/default/1/expansionnet_v2_ieee_bigdata2023/ExpansionNet_v2_src/models/captioning_model.py:206: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  new_loop_dec_classes = loop_dec_classes[[bs_idxes, which_sequence]]
/kaggle/input/models/nguynthnhdan/expansionnet-v2-ieee-bigdata2023/pytorch/default/1/expansionnet_v2_ieee_bigdata2023/ExpansionNet_v2_src/models/captioning_model.py:207: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor

PHASE 03C EXPANSIONNET V2 CELL 5 STATUS
Batch 1 : PASSED {'batch_size': 1, 'status': 'passed', 'elapsed_seconds': 1.1547491870001068, 'seconds_per_image': 1.1547491870001068, 'peak_gpu_memory_mib': 1041.95751953125}
Batch 2 : PASSED {'batch_size': 2, 'status': 'passed', 'elapsed_seconds': 0.3038263519999873, 'seconds_per_image': 0.15191317599999365, 'peak_gpu_memory_mib': 1167.8330078125}
Batch 4 : PASSED {'batch_size': 4, 'status': 'passed', 'elapsed_seconds': 0.41284336800003985, 'seconds_per_image': 0.10321084200000996, 'peak_gpu_memory_mib': 1419.2080078125}
Batch 8 : PASSED {'batch_size': 8, 'status': 'passed', 'elapsed_seconds': 0.7043236609999894, 'seconds_per_image': 0.08804045762499868, 'peak_gpu_memory_mib': 1906.4580078125}
Selected batch size       : 8
Selected repeat exact     : True
Test Set status           : SEALED
Progress                  : 5 / 7 code cells
Next                      : discarded 64-image production pilot


## Cell 6 — Discarded 64-image production pilot và sharding plan

Sinh 64 pseudo-caption, ánh xạ sang S1 IDs, kiểm resume boundary và ước lượng
thời gian cho 113.287 ảnh. Pilot chỉ dùng audit/examples, không được trộn vào
production pseudo-caption.


In [6]:
CELL_STARTED_AT = time.perf_counter()

PILOT_COUNT = 64
pilot_records = []
pilot_started = time.perf_counter()
for start in range(0, PILOT_COUNT, SELECTED_BATCH):
    stop = min(start + SELECTED_BATCH, PILOT_COUNT)
    samples = [TRAIN_DATASET[index] for index in range(start, stop)]
    images = collate_samples(samples)
    native_ids, captions, _ = generate_batch(images)
    for sample, teacher_ids, caption in zip(
        samples, native_ids, captions
    ):
        lexical, student_ids = caption_to_s1(caption)
        pilot_records.append(
            {
                "image_id": int(sample["image_id"]),
                "caption": caption,
                "teacher_token_ids": teacher_ids,
                "student_token_ids": student_ids,
                "student_lexical_tokens": lexical,
            }
        )
pilot_elapsed = time.perf_counter() - pilot_started
assert len(pilot_records) == PILOT_COUNT
assert len({row["image_id"] for row in pilot_records}) == PILOT_COUNT
assert all(row["caption"] for row in pilot_records)
assert all(
    row["student_token_ids"][0] == 1
    and row["student_token_ids"][-1] == 2
    for row in pilot_records
)

# Repeat the first production batch exactly.
repeat_samples = [
    TRAIN_DATASET[index] for index in range(SELECTED_BATCH)
]
repeat_images = collate_samples(repeat_samples)
repeat_ids, repeat_captions, _ = generate_batch(repeat_images)
assert repeat_ids == [
    row["teacher_token_ids"]
    for row in pilot_records[:SELECTED_BATCH]
]
assert repeat_captions == [
    row["caption"] for row in pilot_records[:SELECTED_BATCH]
]

seconds_per_image = pilot_elapsed / PILOT_COUNT
estimated_full_hours = seconds_per_image * 113287 / 3600
target_hours_per_shard = 6.0
recommended_shards = max(
    1, math.ceil(estimated_full_hours / target_hours_per_shard)
)
shard_size = math.ceil(113287 / recommended_shards)
shards = []
for shard_id in range(recommended_shards):
    start = shard_id * shard_size
    stop = min((shard_id + 1) * shard_size, 113287)
    if start < stop:
        shards.append(
            {
                "shard_id": shard_id,
                "start_index": start,
                "stop_index_exclusive": stop,
                "image_count": stop - start,
                "first_image_id": TRAIN_IMAGE_IDS[start],
                "last_image_id": TRAIN_IMAGE_IDS[stop - 1],
            }
        )
assert sum(row["image_count"] for row in shards) == 113287

write_json(
    PILOT_DIR / "discarded_pilot_examples.json",
    {
        "status": "discarded_pilot_only",
        "not_valid_as_production_pseudolabels": True,
        "records": pilot_records[:16],
    },
)
SHARD_PLAN_PATH = (
    CONFIG_DIR / "teacher_a_expansionnet_v2_pseudocaption_shards.json"
)
write_json(
    SHARD_PLAN_PATH,
    {
        "contract_version": "phase3c_pseudocaption_shards_v1",
        "created_at_utc": now_utc(),
        "total_images": 113287,
        "selected_batch_size": SELECTED_BATCH,
        "pilot_images": PILOT_COUNT,
        "pilot_elapsed_seconds": pilot_elapsed,
        "seconds_per_image": seconds_per_image,
        "estimated_full_hours_single_gpu": estimated_full_hours,
        "target_hours_per_shard": target_hours_per_shard,
        "recommended_shards": len(shards),
        "shards": shards,
        "resume_boundary": "completed batch / atomic JSONL append",
        "test_set_status": "sealed",
    },
)
write_json(
    AUDIT_DIR / "phase3c_cell06_discarded_pilot.json",
    {
        "audit_version": "phase3c_expansionnet_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "pilot_images": PILOT_COUNT,
        "selected_batch_size": SELECTED_BATCH,
        "captions_nonempty": True,
        "student_bos_eos_valid": True,
        "first_batch_repeat_exact": True,
        "seconds_per_image": seconds_per_image,
        "estimated_full_hours_single_gpu": estimated_full_hours,
        "recommended_shards": len(shards),
        "pilot_used_as_production_data": False,
        "full_generation_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03C EXPANSIONNET V2 CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Discarded pilot images    :", PILOT_COUNT)
print("Selected batch size       :", SELECTED_BATCH)
print("Captions nonempty         : True")
print("S1 BOS/EOS valid          : True")
print("First batch repeat exact  : True")
print("Seconds / image           :", round(seconds_per_image, 4))
print(
    "Estimated full hours     :",
    round(estimated_full_hours, 2),
)
print("Recommended shards        :", len(shards))
print("Pilot used in production  : False")
print("Full generation started   : False")
print("Test Set status           : SEALED")
print("Progress                  : 6 / 7 code cells")
print("Next                      : production contract and READY marker")


PHASE 03C EXPANSIONNET V2 CELL 6 STATUS
Status                    : PASSED
Discarded pilot images    : 64
Selected batch size       : 8
Captions nonempty         : True
S1 BOS/EOS valid          : True
First batch repeat exact  : True
Seconds / image           : 0.1013
Estimated full hours     : 3.19
Recommended shards        : 1
Pilot used in production  : False
Full generation started   : False
Test Set status           : SEALED
Progress                  : 6 / 7 code cells
Next                      : production contract and READY marker


## Cell 7 — Production contract, manifest và Phase 03D authorization

Khóa atomic JSONL, resume cursor, shard boundaries và schema pseudo-caption.
Phase 03D mới được phép sinh đầy đủ 113.287 pseudo-caption.


In [7]:
CELL_STARTED_AT = time.perf_counter()

PRODUCTION_CONTRACT = (
    CONFIG_DIR
    / "teacher_a_expansionnet_v2_pseudocaption_production_contract.json"
)
write_json(
    PRODUCTION_CONTRACT,
    {
        "contract_version": "phase3c_pseudocaption_production_v1",
        "created_at_utc": now_utc(),
        "teacher": "ExpansionNet v2",
        "teacher_checkpoint_sha256": (
            phase3b_ready["checkpoint_sha256"]
        ),
        "teacher_frozen": True,
        "dataset_split": "Karpathy Train+Restval",
        "total_images": 113287,
        "image_order": "ascending image_id",
        "generation": {
            "beam_size": 3,
            "length_penalty": 0.7,
            "maximum_decoder_steps": 31,
            "selected_batch_size": SELECTED_BATCH,
            "deterministic": True,
        },
        "record_schema": {
            "image_id": "int",
            "caption": "str",
            "teacher_token_ids": "list[int]",
            "student_token_ids": "list[int]",
            "teacher_checkpoint_sha256": "str",
            "shard_id": "int",
        },
        "student_mapping_contract": (
            TOKEN_MAPPING_CONTRACT.name
        ),
        "shard_plan": SHARD_PLAN_PATH.name,
        "atomic_output": (
            "write .tmp JSONL + fsync + os.replace at shard completion"
        ),
        "resume": (
            "resume only after last complete JSONL record; verify unique "
            "ascending image IDs and all prior record hashes"
        ),
        "merge_gate": (
            "exactly 113287 unique image IDs, no train/validation/test "
            "overlap, every student sequence begins BOS and ends EOS"
        ),
        "direct_logit_kd_allowed": False,
        "sequence_kd_ready": True,
        "full_generation_started": False,
        "test_set_status": "sealed",
    },
)

artifact_paths = sorted(
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
    and path.name not in {
        "phase3c_expansionnet_v2_readiness_manifest.csv",
        "SHA256SUMS",
        "PHASE3_TEACHER_A_EXPANSIONNET_V2_"
        "PSEUDOCAPTION_READINESS_READY.json",
    }
)
MANIFEST_CSV = (
    MANIFEST_DIR / "phase3c_expansionnet_v2_readiness_manifest.csv"
)
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    for artifact in artifact_paths:
        writer.writerow(
            {
                "relative_path": artifact.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "size_bytes": artifact.stat().st_size,
                "sha256": sha256_file(artifact),
            }
        )
with MANIFEST_CSV.open(
    "r", encoding="utf-8", newline=""
) as handle:
    manifest_rows = list(csv.DictReader(handle))
SHA256SUMS = MANIFEST_DIR / "SHA256SUMS"
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

READY_PATH = (
    OUTPUT_ROOT
    / "PHASE3_TEACHER_A_EXPANSIONNET_V2_"
    "PSEUDOCAPTION_READINESS_READY.json"
)
write_json(
    READY_PATH,
    {
        "schema_version": "phase3c_expansionnet_readiness_ready_v1",
        "created_at_utc": now_utc(),
        "status": "ready",
        "phase": "03C",
        "teacher": "ExpansionNet v2",
        "matched_student": "S1 MobileNetV3-Large",
        "phase3b_validation_metrics": (
            phase3b_ready["validation_metrics"]
        ),
        "selected_batch_size": SELECTED_BATCH,
        "pilot_images": PILOT_COUNT,
        "pilot_repeat_exact": True,
        "estimated_full_hours_single_gpu": estimated_full_hours,
        "recommended_shards": len(shards),
        "direct_logit_kd_allowed": False,
        "sequence_kd_ready": True,
        "phase03d_full_pseudocaption_generation_authorized": True,
        "production_contract": {
            "relative_path": PRODUCTION_CONTRACT.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(PRODUCTION_CONTRACT),
        },
        "token_mapping_contract": {
            "relative_path": TOKEN_MAPPING_CONTRACT.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(TOKEN_MAPPING_CONTRACT),
        },
        "shard_plan": {
            "relative_path": SHARD_PLAN_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(SHARD_PLAN_PATH),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
                "rows": len(manifest_rows),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "full_generation_started": False,
        "training_started": False,
        "test_parquet_opened": False,
        "test_caption_read": False,
        "test_images_read": False,
        "test_metrics_computed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 03C EXPANSIONNET V2 FINAL STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher                   : A — ExpansionNet v2")
print("Matched Student           : S1 — MobileNetV3-Large")
print("Phase 03B quality gate    : VERIFIED")
print("Train+Restval images      : 113287")
print("Selected batch size       :", SELECTED_BATCH)
print("Recommended shards        :", len(shards))
print("Pilot repeat exact        : True")
print("Direct Logit KD           : PROHIBITED")
print("Sequence KD engine        : READY")
print("Full generation started   : False")
print("Training started          : False")
print("Manifest rows             :", len(manifest_rows), "VERIFIED")
print("Test Set status           : SEALED")
print("READY marker              :", READY_PATH)
print("READY SHA-256             :", sha256_file(READY_PATH))
print("Progress                  : 7 / 7 code cells")
print(
    "Next                      : Phase 03D full Train+Restval "
    "pseudo-caption generation by authorized shards"
)


PHASE 03C EXPANSIONNET V2 FINAL STATUS
Status                    : PASSED
Teacher                   : A — ExpansionNet v2
Matched Student           : S1 — MobileNetV3-Large
Phase 03B quality gate    : VERIFIED
Train+Restval images      : 113287
Selected batch size       : 8
Recommended shards        : 1
Pilot repeat exact        : True
Direct Logit KD           : PROHIBITED
Sequence KD engine        : READY
Full generation started   : False
Training started          : False
Manifest rows             : 11 VERIFIED
Test Set status           : SEALED
READY marker              : /kaggle/working//phase3_teacher_a_expansionnet_v2_readiness/PHASE3_TEACHER_A_EXPANSIONNET_V2_PSEUDOCAPTION_READINESS_READY.json
READY SHA-256             : 930f550fc326b3cf188b823e85b9bfaadc4a46407ad3f414a6b6f7df54770aae
Progress                  : 7 / 7 code cells
Next                      : Phase 03D full Train+Restval pseudo-caption generation by authorized shards
